# LSTM Sanity Checks (CLASSIFIER_v2)

Runs four ablations on the *same* held-out test set against an already-trained GELSTM checkpoint, and tabulates them side-by-side:

1. **Baseline** — original config (`use_time_delta=True`, full visit history, true temporal order).
2. **Shuffled visit order** — visits per subject randomly permuted at eval. Δt rides with its original visit so the Δt distribution is preserved while order is destroyed.
3. **Δt removed** — `use_time_delta=False` at eval.
4. **Fixed sequence length (first-2 visits)** — uses `LongitudinalSubjectDataset(max_visits=2, require_full_window=True)`.
5. **Metadata-only baseline** — copied across from `SANITY_TIME_METADATA_BASELINE.ipynb` for cross-reference.

*All four LSTM rows use the same trained checkpoint to isolate the effect of each ablation.*

In [ ]:
import sys
from pathlib import Path
import json, os, copy
import numpy as np
import pandas as pd
import torch

V2_ROOT = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER_v2')
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from model.GELSTM.models  import GELSTMClassifier
from model.GELSTM.dataset import LongitudinalSubjectDataset
from model.GELSTM.train   import evaluate, make_batches
from model.GELSTM.utils   import set_seed
from common.sanity        import run_full_audit

RANDOM_STATE = 42
set_seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
DATA_VERSION = '__v3__'
DATA_ROOT    = Path('/mnt/e/fyassine/ad-early-detection/DATA/DELCODE') / DATA_VERSION
WB_DATA_ROOT = str(DATA_ROOT / 'matrices')
METADATA_DIR = DATA_ROOT / 'metadata'
COHORTS_CSV  = str(METADATA_DIR / 'cohorts.csv')

SPLIT_CSVS = {
    'train': str(METADATA_DIR / 'splits_gaae' / 'train.csv'),
    'val':   str(METADATA_DIR / 'splits_gaae' / 'val.csv'),
    'test':  str(METADATA_DIR / 'splits_gaae' / 'test.csv'),
}
_ = run_full_audit(SPLIT_CSVS)

# Point this at the GELSTM checkpoint to be audited.
GELSTM_CKPT_PATH = ''   # e.g. '/mnt/e/.../checkpoints_gelstm_whole_brain/<run>/model_<run>.pth'
GAAE_CKPT_PATH   = ''   # GAAE encoder weights bundled into the GELSTM checkpoint
GAAE_CONFIG_PATH = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER/configs/gaae_delcode_whole_brain.json')

if not GELSTM_CKPT_PATH:
    print('NOTE: set GELSTM_CKPT_PATH and GAAE_CKPT_PATH before running cells below.')

## Load test set + checkpoint

We build two test datasets: one with **all visits** (used by checks 1-3), one with **first-2-visits and require_full_window=True** (used by check 4).

In [ ]:
test_df = pd.read_csv(SPLIT_CSVS['test'])
test_mci = test_df[test_df['diagnosis'].isin(['mci', 'converter'])].copy()
print(f'Test subjects (MCI/converter): {len(test_mci)}')

with open(GAAE_CONFIG_PATH) as f:
    hp = json.load(f)
ADJ_K        = hp.get('adjacency_k', 8)
FILE_VARIANT = hp.get('file_variant', 'z_transformed')

test_ds_all = LongitudinalSubjectDataset(
    WB_DATA_ROOT, test_mci, COHORTS_CSV,
    adjacency_k=ADJ_K, file_variant=FILE_VARIANT,
)
test_ds_first2 = LongitudinalSubjectDataset(
    WB_DATA_ROOT, test_mci, COHORTS_CSV,
    adjacency_k=ADJ_K, file_variant=FILE_VARIANT,
    max_visits=2, require_full_window=True,
)
test_items_all    = [test_ds_all[i]    for i in range(len(test_ds_all))]
test_items_first2 = [test_ds_first2[i] for i in range(len(test_ds_first2))]

In [ ]:
def load_gelstm():
    m = GELSTMClassifier(
        in_features=200, gaae_hidden=200,
        gaae_latent=hp.get('latent_dim', 64),
        gaae_heads=hp.get('num_heads', 2),
        gaae_cond_dim=hp.get('cond_dim', 2),
        gaae_dropout=hp.get('dropout', 0.3),
        lstm_hidden=128, lstm_layers=2, lstm_dropout=0.3,
        use_time_delta=True, classifier_hidden=64,
    ).to(device)
    state = torch.load(GELSTM_CKPT_PATH, map_location=device)
    # Accept either a bare state_dict or a checkpoint dict.
    if isinstance(state, dict) and 'state_dict' in state:
        state = state['state_dict']
    m.load_state_dict(state, strict=False)
    m.eval()
    return m

if GELSTM_CKPT_PATH:
    model = load_gelstm()
    print('Model loaded.')

## Run the four ablations

In [ ]:
BATCH_SIZE = 16
rng = np.random.default_rng(RANDOM_STATE)
rows = []

def _run(items, **eval_kwargs):
    batches = make_batches(items, BATCH_SIZE, shuffle=False)
    return evaluate(model, batches, device, **eval_kwargs)

if GELSTM_CKPT_PATH:
    # 1. Baseline.
    r = _run(test_items_all, use_time_delta=True, shuffle_order=False)
    rows.append({'check': '1_baseline', 'auc': r['auc'], 'sens': r['sensitivity'], 'spec': r['specificity']})

    # 2. Shuffled visit order (repeat 5x to estimate variance from the random permutation).
    aucs = []
    for k in range(5):
        r = _run(test_items_all, use_time_delta=True, shuffle_order=True,
                 shuffle_rng=np.random.default_rng(RANDOM_STATE + k))
        aucs.append(r['auc'])
    rows.append({'check': '2_shuffled_order', 'auc': float(np.mean(aucs)), 'sens': float('nan'),
                 'spec': float('nan'), 'auc_std': float(np.std(aucs))})

    # 3. Δt removed.
    r = _run(test_items_all, use_time_delta=False, shuffle_order=False)
    rows.append({'check': '3_no_delta_t', 'auc': r['auc'], 'sens': r['sensitivity'], 'spec': r['specificity']})

    # 4. Fixed sequence length (first 2 visits, full-window).
    r = _run(test_items_first2, use_time_delta=True, shuffle_order=False)
    rows.append({'check': '4_first_2_visits', 'auc': r['auc'], 'sens': r['sensitivity'],
                 'spec': r['specificity'], 'n_subjects': len(test_items_first2)})

pd.DataFrame(rows)

## Cross-reference: metadata-only baseline

Reuse the result table from `SANITY_TIME_METADATA_BASELINE.ipynb` to anchor row 5 of the sanity table — set the value manually after running that notebook.

In [ ]:
META_ONLY_CV_AUC = float('nan')   # paste from SANITY_TIME_METADATA_BASELINE.ipynb
META_ONLY_TEST_AUC = float('nan')
rows.append({
    'check': '5_metadata_only_baseline',
    'auc':    META_ONLY_TEST_AUC,
    'cv_auc': META_ONLY_CV_AUC,
})
summary = pd.DataFrame(rows)
summary

## Reading the table

* **Baseline vs shuffled-order.** If row 2 ≈ row 1, the LSTM is *not* exploiting temporal order — the per-visit graph embedding is doing the work, the recurrence is decorative.
* **Baseline vs Δt-removed.** If row 3 collapses, Δt is carrying real signal (which may itself be leakage; see metadata baseline).
* **Baseline vs first-2-visits.** If row 4 stays high while only seeing the *earliest* two scans, the model is doing *true early detection*. If it collapses, prior numbers were aided by later visits or by sequence-length cues.
* **Metadata-only.** Anchor for everything: subtract this from every fMRI-based row to estimate the marginal contribution of brain features.